In [1]:
import torch.nn as nn
from torch.nn import functional as F
from AlignmentTechniques import LatentAlignment2d, AdaptiveBatchNorm2d, EuclideanAlignment


class DeepSleep(nn.Module):
    """
    DeepSleep implementation  based on
    Chambon, S., Galtier, M.N., Arnal, P.J., Wainrib, G. and Gramfort, A., 2018.
    A deep learning architecture for temporal sleep stage classification using multivariate and multimodal time series.
    IEEE Transactions on Neural Systems and Rehabilitation Engineering, 26(4), pp.758-769.
    https://ieeexplore.ieee.org/document/8307462.
    """

    def __init__(self, in_shape, n_out, alignment='None', dropout=0.25):
        super(DeepSleep, self).__init__()
        self.in_shape = in_shape
        self.n_out = n_out
        self.alignment = alignment

        n_filters = 2
        n_spatial = 8
        self.n_filters = n_filters
        self.n_spatial = n_spatial

        if alignment == 'euclidean':
            self.euclidean = EuclideanAlignment()
        # Input bn
        if alignment == 'latent':
            self.latent_align0 = LatentAlignment2d(in_shape[0], affine=False)
        elif alignment == 'adaptive':
            self.abn0 = AdaptiveBatchNorm2d(in_shape[0], affine=False)
        else:
            self.bn0 = nn.BatchNorm2d(in_shape[0], affine=False)

        # Spatial filters
        self.conv1 = nn.Conv2d(1, n_spatial,
                               kernel_size=(in_shape[0], 1),
                               bias=True)
        if alignment == 'latent':
            self.latent_align1 = LatentAlignment2d(self.conv1.out_channels)
        elif alignment == 'adaptive':
            self.abn1 = AdaptiveBatchNorm2d(self.conv1.out_channels)
        else:
            self.bn1 = nn.BatchNorm2d(self.conv1.out_channels)

        # Block 1
        self.conv2 = nn.Conv2d(n_spatial, n_spatial * n_filters,
                               kernel_size=(1, 51), padding=(0, 25),
                               bias=True, groups=1)
        if alignment == 'latent':
            self.latent_align2 = LatentAlignment2d(self.conv2.out_channels)
        elif alignment == 'adaptive':
            self.abn2 = AdaptiveBatchNorm2d(self.conv2.out_channels)
        else:
            self.bn2 = nn.BatchNorm2d(self.conv2.out_channels)
        self.pool1 = nn.MaxPool2d(kernel_size=(1, 16))

        # Block 2
        self.conv3 = nn.Conv2d(self.conv2.out_channels, self.conv2.out_channels,
                               kernel_size=(1, 51), padding=(0, 25),
                               bias=True, groups=1)
        if alignment == 'latent':
            self.latent_align3 = LatentAlignment2d(self.conv2.out_channels)
        elif alignment == 'adaptive':
            self.abn3 = AdaptiveBatchNorm2d(self.conv2.out_channels)
        else:
            self.bn3 = nn.BatchNorm2d(self.conv2.out_channels)
        self.pool2 = nn.MaxPool2d(kernel_size=(1, 16))
        self.drop1 = nn.Dropout(dropout)

        # Classifier
        self.n_features = int(self.conv3.out_channels) * (in_shape[-1] // 16 // 16)
        self.fc_out = nn.Linear(self.n_features, n_out)

    def forward(self, x, sbj_trials):
        """
        Args:
             x: (batch * sbj_trials, spatial, time)
             sbj_trials: number of trials per subject
        """
        _, spatial, time = x.shape

        # Euclidean alignment
        if self.alignment == 'euclidean':
            x = self.euclidean(x, sbj_trials)

        # Input batchnorm
        x = x.reshape(-1, spatial, 1, time)
        if self.alignment == 'latent':
            x = self.latent_align0(x, sbj_trials)
        elif self.alignment == 'adaptive':
            x = self.abn0(x, sbj_trials)
        else:
            x = self.bn0(x)
        x = x.reshape(-1, spatial, time)

        # Add artificial image channel dimension for Conv2d
        x = x.unsqueeze(1)

        # Spatial
        x = self.conv1(x)  # (batch * sbj, spatial, 1, time)
        if self.alignment == 'latent':
            # x = self.latent_align1(x, sbj_trials, growing_context=growing_context) errore la variabile growing_context non è definita da nessuna parte
            x = self.latent_align1(x, sbj_trials)
        elif self.alignment == 'adaptive':
            x = self.abn1(x, sbj_trials)
        else:
            x = self.bn1(x)

        # Block 1
        x = self.conv2(x)  # (batch * sbj, filters, spatial, time)
        if self.alignment == 'latent':
            x = self.latent_align2(x, sbj_trials)
        elif self.alignment == 'adaptive':
            x = self.abn2(x, sbj_trials)
        else:
            x = self.bn2(x)
        x = F.relu(x)
        x = self.pool1(x)

        # Block 2
        x = self.conv3(x)
        if self.alignment == 'latent':
            x = self.latent_align3(x, sbj_trials)
        elif self.alignment == 'adaptive':
            x = self.abn3(x, sbj_trials)
        else:
            x = self.bn3(x)
        x = self.pool2(x)
        x = self.drop1(x)

        # Classifier
        x = x.reshape(-1, self.n_features)
        x = F.relu(x)
        x = self.fc_out(x)

        return x


In [2]:
import mne
import numpy as np
import pandas as pd
import gc
import torch
import torch.optim as optim
import torch.nn as nn
from sklearn.metrics import accuracy_score
import time
# --- FUNZIONE PER ESTRARRE I DATI DEL SONNO (Sleep-EDF) ---
def load_sleep_data(subject_id):
    # Scarica i dati per il soggetto (usa la registrazione 1)
    files = mne.datasets.sleep_physionet.age.fetch_data(subjects=[subject_id], recording=[1])[0]
    raw = mne.io.read_raw_edf(files[0], preload=True)
    annot = mne.read_annotations(files[1])
    raw.set_annotations(annot, emit_warning=False)
    
    # Selezioniamo solo i canali EEG 
    # Questo evita di portarsi dietro EOG, EMG o altri 5 canali inutili
    raw.pick_channels(['EEG Fpz-Cz', 'EEG Pz-Oz'])
    raw.resample(100.0) # Il paper DeepSleep assume 100Hz
    
    # Mappatura degli stadi del sonno in 5 classi
    mapping = {'Sleep stage W': 1, 'Sleep stage 1': 2, 'Sleep stage 2': 3,
               'Sleep stage 3': 4, 'Sleep stage 4': 4, 'Sleep stage R': 5}
    
    # Creiamo gli eventi ogni 30 secondi
    events, _ = mne.events_from_annotations(raw, event_id=mapping, chunk_duration=30.)
    
    # Tagliamo i dati in epoche di 30 secondi (3000 campioni a 100Hz)
    tmax = 30. - 1. / raw.info['sfreq']
    epochs = mne.Epochs(raw=raw, events=events, event_id=[1, 2, 3, 4, 5],
                        tmin=0., tmax=tmax, baseline=None, preload=True)
    
    X = epochs.get_data(copy=True) * 1e6 # Convertiamo in microvolts per la rete neurale
    y = epochs.events[:, 2] - 1 # Scaliamo le classi da 0 a 4
    
    return X, y

print("Librerie e funzioni caricate con successo!")

Librerie e funzioni caricate con successo!


In [3]:
import os

# --- CONFIGURAZIONE POTENZIATA ---
num_cores = os.cpu_count()
torch.set_num_threads(num_cores)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Sistema: {device} | Core CPU pronti: {num_cores}")

Sistema: cpu | Core CPU pronti: 6


In [4]:
lista_soggetti = list(range(5)) # PhysioNet Sleep-EDF usa indici a partire da 0
X_list, y_list, meta_list = [], [], []

print("Estrazione dati Sleep-EDF in corso (potrebbe richiedere un minuto per il download)...")
for sbj in lista_soggetti:
    print(f"Elaborazione Soggetto {sbj}...")
    X_tmp, y_tmp = load_sleep_data(sbj)
    
    X_list.append(X_tmp.astype('float32'))
    y_list.append(y_tmp)
    # Creiamo i metadata manualmente visto che non usiamo MOABB
    meta_list.append(pd.DataFrame({'subject': [sbj] * len(y_tmp)}))
    
    del X_tmp
    gc.collect()

X = np.concatenate(X_list, axis=0)
y = np.concatenate(y_list, axis=0)
metadata = pd.concat(meta_list, ignore_index=True)

print(f"\n✅ Caricamento completato!")
print(f"Shape X: {X.shape} -> (prove, 2 canali, 3000 campioni)")

Estrazione dati Sleep-EDF in corso (potrebbe richiedere un minuto per il download)...
Elaborazione Soggetto 0...
Extracting EDF parameters from /home/devcontainers/mne_data/physionet-sleep-data/SC4001E0-PSG.edf...
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 7949999  =      0.000 ... 79499.990 secs...


/home/devcontainers/miniconda3/envs/eeg_clean/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/tmp/ipykernel_69308/2848585093.py:14: RuntimeWarning: Channels contain different highpass filters. Highest filter setting will be stored.
  raw = mne.io.read_raw_edf(files[0], preload=True)
/tmp/ipykernel_69308/2848585093.py:14: RuntimeWarning: Channels contain different lowpass filters. Lowest filter setting will be stored.
  raw = mne.io.read_raw_edf(files[0], preload=True)
/tmp/ipykernel_69308/2848585093.py:14: RuntimeWarning: Highpass cutoff frequency 16.0 is greater than lowpass cutoff frequency 0.7, setting values to 0 and Nyquist.
  raw = mne.io.read_raw_edf(files[0], preload=True)


NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).
Sampling frequency of the instance is already 100.0, returning unmodified.
Used Annotations descriptions: [np.str_('Sleep stage 1'), np.str_('Sleep stage 2'), np.str_('Sleep stage 3'), np.str_('Sleep stage 4'), np.str_('Sleep stage R'), np.str_('Sleep stage W')]
Not setting metadata
2650 matching events found
No baseline correction applied
0 projection items activated
Using data from preloaded Raw for 2650 events and 3000 original time points ...
0 bad epochs dropped
Elaborazione Soggetto 1...
Extracting EDF parameters from /home/devcontainers/mne_data/physionet-sleep-data/SC4011E0-PSG.edf...
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 8405999  =      0.000 ... 84059.990 secs...


/tmp/ipykernel_69308/2848585093.py:14: RuntimeWarning: Channels contain different highpass filters. Highest filter setting will be stored.
  raw = mne.io.read_raw_edf(files[0], preload=True)
/tmp/ipykernel_69308/2848585093.py:14: RuntimeWarning: Channels contain different lowpass filters. Lowest filter setting will be stored.
  raw = mne.io.read_raw_edf(files[0], preload=True)
/tmp/ipykernel_69308/2848585093.py:14: RuntimeWarning: Highpass cutoff frequency 16.0 is greater than lowpass cutoff frequency 0.7, setting values to 0 and Nyquist.
  raw = mne.io.read_raw_edf(files[0], preload=True)


NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).
Sampling frequency of the instance is already 100.0, returning unmodified.
Used Annotations descriptions: [np.str_('Sleep stage 1'), np.str_('Sleep stage 2'), np.str_('Sleep stage 3'), np.str_('Sleep stage 4'), np.str_('Sleep stage R'), np.str_('Sleep stage W')]
Not setting metadata
2802 matching events found
No baseline correction applied
0 projection items activated
Using data from preloaded Raw for 2802 events and 3000 original time points ...
0 bad epochs dropped
Elaborazione Soggetto 2...
Extracting EDF parameters from /home/devcontainers/mne_data/physionet-sleep-data/SC4021E0-PSG.edf...
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 8411999  =      0.000 ... 84119.990 secs...


...

NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).
Sampling frequency of the instance is already 100.0, returning unmodified.
Used Annotations descriptions: [np.str_('Sleep stage 1'), np.str_('Sleep stage 2'), np.str_('Sleep stage 3'), np.str_('Sleep stage 4'), np.str_('Sleep stage R'), np.str_('Sleep stage W')]
Not setting metadata
2804 matching events found
No baseline correction applied
0 projection items activated
Using data from preloaded Raw for 2804 events and 3000 original time points ...
0 bad epochs dropped
Elaborazione Soggetto 3...
Extracting EDF parameters from /home/devcontainers/mne_data/physionet-sleep-data/SC4031E0-PSG.edf...
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 8459999  =      0.000 ... 84599.990 secs...


/tmp/ipykernel_69308/2848585093.py:14: RuntimeWarning: Channels contain different highpass filters. Highest filter setting will be stored.
  raw = mne.io.read_raw_edf(files[0], preload=True)
/tmp/ipykernel_69308/2848585093.py:14: RuntimeWarning: Channels contain different lowpass filters. Lowest filter setting will be stored.
  raw = mne.io.read_raw_edf(files[0], preload=True)
/tmp/ipykernel_69308/2848585093.py:14: RuntimeWarning: Highpass cutoff frequency 16.0 is greater than lowpass cutoff frequency 0.7, setting values to 0 and Nyquist.
  raw = mne.io.read_raw_edf(files[0], preload=True)


NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).
Sampling frequency of the instance is already 100.0, returning unmodified.
Used Annotations descriptions: [np.str_('Sleep stage 1'), np.str_('Sleep stage 2'), np.str_('Sleep stage 3'), np.str_('Sleep stage 4'), np.str_('Sleep stage R'), np.str_('Sleep stage W')]
Not setting metadata
2820 matching events found
No baseline correction applied
0 projection items activated
Using data from preloaded Raw for 2820 events and 3000 original time points ...
0 bad epochs dropped
Elaborazione Soggetto 4...
Extracting EDF parameters from /home/devcontainers/mne_data/physionet-sleep-data/SC4041E0-PSG.edf...
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 7709999  =      0.000 ... 77099.990 secs...


/tmp/ipykernel_69308/2848585093.py:14: RuntimeWarning: Channels contain different highpass filters. Highest filter setting will be stored.
  raw = mne.io.read_raw_edf(files[0], preload=True)
/tmp/ipykernel_69308/2848585093.py:14: RuntimeWarning: Channels contain different lowpass filters. Lowest filter setting will be stored.
  raw = mne.io.read_raw_edf(files[0], preload=True)
/tmp/ipykernel_69308/2848585093.py:14: RuntimeWarning: Highpass cutoff frequency 16.0 is greater than lowpass cutoff frequency 0.7, setting values to 0 and Nyquist.
  raw = mne.io.read_raw_edf(files[0], preload=True)


NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).
Sampling frequency of the instance is already 100.0, returning unmodified.
Used Annotations descriptions: [np.str_('Sleep stage 1'), np.str_('Sleep stage 2'), np.str_('Sleep stage 3'), np.str_('Sleep stage R'), np.str_('Sleep stage W')]
Not setting metadata
2569 matching events found
No baseline correction applied
0 projection items activated
Using data from preloaded Raw for 2569 events and 3000 original time points ...
0 bad epochs dropped

✅ Caricamento completato!
Shape X: (13645, 2, 3000) -> (prove, 2 canali, 3000 campioni)


In [27]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Utilizzando il device: {device}")

X_tensor = torch.from_numpy(X).float()
y_tensor = torch.from_numpy(y).long()

n_channels = X.shape[1]
n_samples = X.shape[2]
n_classes = len(np.unique(y))

print("\n=== Inizio Validazione LOSO su DeepSleep ===")
loso_results = []
subjects_array = metadata['subject'].unique()
num_epochs = 10 # 10 epoche bastano per vedere se il modello impara

for test_subject in subjects_array:
    print(f"\n-> Addestramento per Test Subject: {test_subject}")
    
    train_mask = (metadata['subject'] != test_subject).values
    test_mask = (metadata['subject'] == test_subject).values
    
    # Inizializzazione DeepSleep con Allineamento Latente
    model = DeepSleep(in_shape=(n_channels, n_samples), n_out=n_classes, alignment='latent').to(device)
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    criterion = nn.CrossEntropyLoss()
    
    train_subjects = metadata[train_mask]['subject'].unique()
    
    model.train()
    start_time = time.time()
    
    for epoch in range(num_epochs):
        epoch_loss = 0.0
        for subj in train_subjects:
            subj_mask = (metadata['subject'] == subj).values
            batch_X = X_tensor[subj_mask].to(device)
            batch_y = y_tensor[subj_mask].to(device)
            
            optimizer.zero_grad(set_to_none=True)
            
            # Deep Sets: passiamo tutte le prove del soggetto (sono circa 900/1000 per il sonno!)
            outputs = model(batch_X, sbj_trials=batch_X.shape[0])
            loss = criterion(outputs, batch_y)
            loss.backward()
            optimizer.step()
            
            epoch_loss += loss.item()
            
        if (epoch + 1) % 5 == 0:
            print(f"   Epoca [{epoch+1}/{num_epochs}], Loss: {epoch_loss/len(train_subjects):.4f}")
            
    # Valutazione sul soggetto escluso
    model.eval()
    with torch.no_grad():
        test_X = X_tensor[test_mask]
        test_y = y_tensor[test_mask]
        outputs = model(test_X, sbj_trials=test_X.shape[0])
        _, predicted = torch.max(outputs.data, 1)
        
        acc = accuracy_score(test_y.cpu().numpy(), predicted.cpu().numpy())
        loso_results.append(acc)
        
    print(f"-> Fine Test Subject {test_subject} | Accuracy: {acc*100:.2f}% | Tempo: {time.time()-start_time:.1f}s")

print(f"\nRISULTATI FINALI LOSO DeepSleep (Latent Alignment): Media {np.mean(loso_results)*100:.2f}%")

Utilizzando il device: cpu

=== Inizio Validazione LOSO su DeepSleep ===

-> Addestramento per Test Subject: 0
   Epoca [5/10], Loss: 0.5747
   Epoca [10/10], Loss: 0.4354
-> Fine Test Subject 0 | Accuracy: 83.58% | Tempo: 420.5s

-> Addestramento per Test Subject: 1
   Epoca [5/10], Loss: 0.5054
   Epoca [10/10], Loss: 0.3708
-> Fine Test Subject 1 | Accuracy: 86.01% | Tempo: 393.5s

-> Addestramento per Test Subject: 2
   Epoca [5/10], Loss: 0.5277
   Epoca [10/10], Loss: 0.3854
-> Fine Test Subject 2 | Accuracy: 89.09% | Tempo: 385.4s

-> Addestramento per Test Subject: 3
   Epoca [5/10], Loss: 0.6147
   Epoca [10/10], Loss: 0.4739
-> Fine Test Subject 3 | Accuracy: 87.20% | Tempo: 384.8s

-> Addestramento per Test Subject: 4
   Epoca [5/10], Loss: 0.5437
   Epoca [10/10], Loss: 0.3824
-> Fine Test Subject 4 | Accuracy: 82.68% | Tempo: 394.5s

RISULTATI FINALI LOSO DeepSleep (Latent Alignment): Media 85.71%


In [ ]:
import torch
import torch.optim as optim
import torch.nn as nn
from sklearn.metrics import accuracy_score
import time

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
X_tensor = torch.from_numpy(X).float()
y_tensor = torch.from_numpy(y).long()

n_channels = X.shape[1]
n_samples = X.shape[2]
n_classes = len(np.unique(y))

print("\n=== Inizio Validazione LOSO su DeepSleep (BASELINE) ===")
loso_results_baseline = []
subjects_array = metadata['subject'].unique()
num_epochs = 10 

for test_subject in subjects_array:
    print(f"\n-> Addestramento per Test Subject: {test_subject}")
    
    train_mask = (metadata['subject'] != test_subject).values
    test_mask = (metadata['subject'] == test_subject).values

    # Preparazione dei dati di training (tutti insieme, non iteriamo per soggetto)
    X_train_loso = X_tensor[train_mask].to(device)
    y_train_loso = y_tensor[train_mask].to(device)
    
    # Inizializzazione DeepSleep SENZA Allineamento Latente
    model = DeepSleep(in_shape=(n_channels, n_samples), n_out=n_classes, alignment='None').to(device)
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    criterion = nn.CrossEntropyLoss()
    
    model.train()
    start_time = time.time()
    
    for epoch in range(num_epochs):
        optimizer.zero_grad(set_to_none=True)
        
        # Passiamo tutto il blocco in una volta (nella baseline sbj_trials non conta)
        outputs = model(X_train_loso, sbj_trials=X_train_loso.shape[0])
        loss = criterion(outputs, y_train_loso)
        loss.backward()
        optimizer.step()
        
        if (epoch + 1) % 5 == 0:
            print(f"   Epoca [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}")
            
    # Valutazione
    model.eval()
    with torch.no_grad():
        test_X = X_tensor[test_mask].to(device)
        test_y = y_tensor[test_mask] # resta in CPU per accuracy_score
        outputs = model(test_X, sbj_trials=test_X.shape[0])
        _, predicted = torch.max(outputs.data, 1)
        
        acc = accuracy_score(test_y.cpu().numpy(), predicted.cpu().numpy())
        loso_results_baseline.append(acc)

    # PULIZIA MEMORIA (importante per Sleep-EDF)
    del X_train_loso, y_train_loso, test_X, model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    print(f"-> Fine Test Subject {test_subject} | Accuracy: {acc*100:.2f}% | Tempo: {time.time()-start_time:.1f}s")

print("\n" + "="*50)
print(f"RISULTATI FINALI LOSO DeepSleep (Baseline)")
print(f"Accuracy Media: {np.mean(loso_results_baseline)*100:.2f}% ± {np.std(loso_results_baseline)*100:.2f}%")
print("="*50)

### 📊 Risultati Esperimento: DeepSleep Baseline

```text
=== Inizio Validazione LOSO su DeepSleep (BASELINE) ===

-> Addestramento per Test Subject: 0
   Epoca [5/10], Loss: 0.6666
   Epoca [10/10], Loss: 0.5751
-> Fine Test Subject 0 | Accuracy: 75.66% | Tempo: 22.4s

-> Addestramento per Test Subject: 1
   Epoca [5/10], Loss: 0.7842
   Epoca [10/10], Loss: 0.6429
-> Fine Test Subject 1 | Accuracy: 74.77% | Tempo: 21.6s

-> Addestramento per Test Subject: 2
   Epoca [5/10], Loss: 0.8295
   Epoca [10/10], Loss: 0.6651
-> Fine Test Subject 2 | Accuracy: 73.75% | Tempo: 22.6s

-> Addestramento per Test Subject: 3
   Epoca [5/10], Loss: 0.8478
   Epoca [10/10], Loss: 0.6610
-> Fine Test Subject 3 | Accuracy: 72.45% | Tempo: 23.8s

-> Addestramento per Test Subject: 4
   Epoca [5/10], Loss: 0.8590
   Epoca [10/10], Loss: 0.6646
-> Fine Test Subject 4 | Accuracy: 59.71% | Tempo: 25.6s

==================================================
RISULTATI FINALI LOSO DeepSleep (Baseline)
Accuracy Media: 71.27% ± 5.88%
==================================================